# 02 · Pipeline walkthrough (demo notebook)

Runs the whole MLOps loop from one place — handy for rehearsing the live demo.
Each step is a script in `src/`, so CI and teammates run *exactly* the same code.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "src").is_dir() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
Path.cwd()

PosixPath('/home/jovyan')

In [2]:
!python -m src.data --kaggle

UCI Wine Quality (red + white Vinho Verde)
  winequality-red.csv already present


  winequality-white.csv already present
  6497 rows, 1177 exact duplicates removed -> 5320
  tiers: {'good': 2323, 'standard': 1988, 'premium': 1009}
  -> /home/jovyan/data/processed/wine_quality.csv (sha aa35c5597a8d)

Kaggle Wine Reviews -> Portuguese catalog
  winemag-data-130k-v2.csv already present


  5256 Portuguese wines | colours: {'red': 3302, 'white': 1481, 'rosé': 239, 'unknown': 234}


  -> /home/jovyan/data/processed/portugal_wines.csv


In [3]:
# Train all configurations -> MLflow http://localhost:5001 (experiment VinhoVerde-Quality)
!python -m src.train

Training 4 configurations on 5320 rows (data sha aa35c5597a8d)



  logreg_baseline  macro-F1 0.543 | acc 0.547 | premium recall 0.738
🏃 View run logreg_baseline at: http://mlflow:5000/#/experiments/1/runs/7fb59917113843ac87242f3ac98d2205
🧪 View experiment at: http://mlflow:5000/#/experiments/1


  rf_balanced      macro-F1 0.599 | acc 0.601 | premium recall 0.649
🏃 View run rf_balanced at: http://mlflow:5000/#/experiments/1/runs/8895e0ce003448579dc69f504a083f0b
🧪 View experiment at: http://mlflow:5000/#/experiments/1


  HistGB           macro-F1 0.563 | acc 0.572 | premium recall 0.465


🏃 View run HistGB at: http://mlflow:5000/#/experiments/1/runs/6427cf2c2875421584a25733ccf540fb
🧪 View experiment at: http://mlflow:5000/#/experiments/1


  rf_unbalanced    macro-F1 0.595 | acc 0.611 | premium recall 0.446
🏃 View run rf_unbalanced at: http://mlflow:5000/#/experiments/1/runs/af6ace47df6f4b6c9b041613229fe9cc
🧪 View experiment at: http://mlflow:5000/#/experiments/1



Best: rf_balanced (macro-F1 0.599). Next: python -m src.register


In [4]:
# Register the best run -> @candidate -> gate -> @champion
!python -m src.register

Best run: rf_balanced (8895e0ce) macro-F1 0.599


Registered model 'VinhoVerde-Quality-Classifier' already exists. Creating a new version of this model...


2026/09/26 22:45:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VinhoVerde-Quality-Classifier, version 3
Created version '3' of model 'VinhoVerde-Quality-Classifier'.


Registered VinhoVerde-Quality-Classifier v3 -> @candidate


Current champion v2 macro-F1 0.599

Candidate v3 macro-F1 0.599
  [PASS] macro_f1 >= 0.55
  [PASS] not worse than champion



@champion -> v3. Tell the API:  curl -X POST http://localhost:8080/reload


In [5]:
import requests
wine = requests.get("http://app:8080/examples").json()["red"]
requests.post("http://app:8080/reload").json(), requests.post("http://app:8080/predict", json=wine).json()

({'reloaded': True, 'model_version': '3'},
 {'quality_tier': 'standard',
  'probabilities': {'good': 0.184, 'premium': 0.028, 'standard': 0.789},
  'action': 'Blend or sell as house wine / bulk. Do not bottle under the estate label.',
  'model_version': '3',
  'input': {'fixed acidity': 7.4,
   'volatile acidity': 0.7,
   'citric acid': 0.0,
   'residual sugar': 1.9,
   'chlorides': 0.076,
   'free sulfur dioxide': 11.0,
   'total sulfur dioxide': 34.0,
   'density': 0.9978,
   'pH': 3.51,
   'sulphates': 0.56,
   'alcohol': 9.4,
   'wine_type': 'red'}})

## LLMOps: prompt versions
~6 minutes on the free tier (5 requests/minute).

In [6]:
!python -m src.evaluate_prompts --promote

Evaluating 3 modes x 11 cases with gemini-3.5-flash-lite (~7 min)



2026/09/26 22:45:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: sommelier, version 7


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


...

🏃 View run grounded_structured at: http://mlflow:5000/#/experiments/2/runs/e823039653634dab8072b971ebab6e70
🧪 View experiment at: http://mlflow:5000/#/experiments/2


  grounded_structured  overall 0.900 | coverage 0.71 | grounding 1.00 | refusals 1.00 | false refusals 0.00



Gate passed for grounded.


@champion -> sommelier v8. Reload: curl -X POST http://localhost:8080/prompt/reload
🏃 View run prompt-mode-comparison at: http://mlflow:5000/#/experiments/2/runs/93f8d288937a46b084b1b94ddc244d87
🧪 View experiment at: http://mlflow:5000/#/experiments/2


2026/09/26 22:51:51 INFO mlflow.tracing.export.async_export_queue: Flushing the async trace logging queue before program exit. This may take a while...


In [7]:
requests.post("http://app:8080/prompt/reload").json()

{'reloaded': True, 'prompt_version': 8, 'mode': 'grounded'}

In [8]:
r = requests.post("http://app:8080/chat", json={"message": "A Douro red under 20 euros for steak?"}).json()
print(r["prompt_mode"], r["prompt_version"]); print(r["answer"])

grounded 8
Ah, a fine steak deserves a wonderful Douro red! From our catalog, I can happily recommend a few fantastic options under €20 that will pair brilliantly with your meal:

* **Caves Velhas 2015 Encostas do Douro Red** (€9) – Rich and spicy with red fruits and soft tannins, offering a wonderful layer of acidity.
* **Quevedo 2014 Oscar's Red** (€15) – Accessible and full of red berries, balanced with acidity and a pleasant layer of tannins.
* **Quinta da Pacheca 2013 Superior Tinto Red** (€14) – Ready to drink, showcasing generous berry flavors, tannins, and a touch of sweetness at the end.
* **Castello d'Alba 2009 Red** (€10) – A juicy wine driven by red fruits and spice, complemented by mineral freshness.
* **Conceito Vinhos 2014 Contraste Tinto Red** (€19) – A slightly more structured choice with dark tannins, concentration, red-berry fruits, and strong spice and wood-aging notes.

Any of these will bring out the best in a good steak! *Saúde!*


## Demo trick: show a promotion live
1. Add a new configuration to `CONFIGS` in `src/train.py` (e.g. more trees).
2. `!python -m src.train --only <new_name>` → `!python -m src.register`
3. Gate passes or blocks it → `/reload` → the UI footer shows the new model version.